# Import Libraries

In [48]:
import pandas as pd
import numpy as np
import spacy
import os
from tqdm import tqdm 
from sklearn.metrics import classification_report
import os
import requests
from openai import OpenAI, RateLimitError
import time
import random

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_colwidth', None)  # Show full content in each cell
pd.set_option('display.width', 1000)  # Set max width

# Load spaCy's English model
nlp = spacy.load('en_core_web_sm')

# Pre-Processing

In [30]:
label_mapper = {
    'knowledge' : 0,
    'comprehension' : 1,
    'application' : 2,
    'analysis' : 3,
    'synthesis' : 4,
    'evaluation' : 5
}

mapping = {
    'knowledge': 'knowledge',
    'remember': 'knowledge',
    'comprehension': 'comprehension',
    'understand': 'comprehension',
    'application': 'application',
    'apply': 'application',
    'analysis': 'analysis',
    'analyse': 'analysis',
    'evaluation': 'evaluation',
    'evaluate': 'evaluation',
    'synthesis': 'synthesis',
    'create': 'synthesis'
}

q_df = pd.read_csv(os.getcwd().replace('notebook' , 'dataset') + '/dataset4.csv')
queries = q_df['question']
q_df['label'] = q_df['label'].str.lower()
q_df['label'] = q_df['label'].replace(mapping)
label = q_df['label'].str.lower().map(label_mapper)
print(q_df['label'].value_counts())

label
synthesis        29
knowledge        22
evaluation       21
comprehension    20
analysis         19
application      15
Name: count, dtype: int64


# API Setup

In [31]:
# Sonar
 
api_key = os.environ.get("PERPLEXITY_API_KEY")

if api_key:
    print('successful')

url = "https://api.perplexity.ai/chat/completions"
headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

successful


In [17]:
# Groq

api_key = os.environ.get("GROQ_API_KEY")

if api_key:
    print('successful')

groq_client = OpenAI(
    base_url = "https://api.groq.com/openai/v1",
    api_key = api_key
)

successful


In [18]:
# OpenRouter

api_key = os.environ.get("OPENROUTER_API_KEY")

if api_key:
    print('successful')

or_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key = api_key
)

successful


# Zero-Shot

## SONAR

In [ ]:
# Test

query = 'How many total disk access is needed to search a record using two level indexing?'
payload = {
    "model": "sonar-pro",
    "messages": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": f"""Classify the query's Bloom's taxonomy level using ONLY one word from: 
            [knowledge, comprehension, application, analysis, synthesis, evaluation]
         
         query : {query}"""}
    ],
    "max_tokens": 100,
    "temperature": 0.5
}
response = requests.post(url, headers=headers, json=payload).json()
reply = response["choices"][0]["message"]["content"]

payload = {
    "model": "sonar-pro",
    "messages": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": f"""Extract the blooms level from my previous reponse. Answer only in one word from: 
            [knowledge, comprehension, application, analysis, synthesis, evaluation]
         previous response : {reply}"""}
    ],
    "max_tokens": 100,
    "temperature": 0.5
}
response = requests.post(url, headers=headers, json=payload).json()
reply = response["choices"][0]["message"]["content"]

print(reply)


### Assign labels

In [ ]:
pred_labels= []

for query in tqdm(queries):
    payload = {
        "model": "sonar-pro",
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": f"""Classify the query based on Bloom's taxonomy level using ONLY one word from: 
                [knowledge, comprehension, application, analysis, synthesis, evaluation]
            
            query : {query}"""}
        ],
        "max_tokens": 100,
        "temperature": 0.5
    }

    response = requests.post(url, headers=headers, json=payload).json()
    reply = response["choices"][0]["message"]["content"]

    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        print(reply)

        payload = {
            "model": "sonar-pro",
            "messages": [
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": f"""Extract the blooms level from previous reponse. Answer only in one word without punctuation from: 
                    [knowledge, comprehension, application, analysis, synthesis, evaluation]
                previous response : {reply}"""}
            ],
            "max_tokens": 100,
            "temperature": 0.5
        }
        response = requests.post(url, headers=headers, json=payload).json()
        reply = response["choices"][0]["message"]["content"]

    pred_labels.append(reply.lower())

In [ ]:
print(classification_report(label , [label_mapper[key.lower()] for key in pred_labels]))

## GPT-OSS-120B

In [ ]:
# Test

query = 'How many total disk access is needed to search a record using two level indexing?'
chat_completion = groq_client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": f"""Classify the query based on Bloom's taxonomy level using ONLY one word from: 
                [knowledge, comprehension, application, analysis, synthesis, evaluation]
            
            query : {query}""",
        }
    ],
    model = "openai/gpt-oss-120b",
)

reply = chat_completion.choices[0].message.content.lower()

while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""Extract the blooms level from previous reponse. Answer only in one word without punctuation from: 
                    [knowledge , comprehension , application , analysis, synthesis , evaluation]
                previous response : {reply}""",
            }
        ],
        model="openai/gpt-oss-120b",
    )

    reply = chat_completion.choices[0].message.content.lower()
    print(reply)


print(reply)

application
application


### Assign Labels

In [58]:
pred_labels = []

for query in tqdm(queries):
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""Classify the query based on Bloom's taxonomy level using ONLY one word from: 
                    [knowledge, comprehension, application, analysis, synthesis, evaluation]
                
                query : {query}""",
            }
        ],
        model="openai/gpt-oss-120b",
    )

    reply = chat_completion.choices[0].message.content.lower()

    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        print(reply)
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": f"""Extract the blooms level from previous reponse. Answer only in one word without punctuation from: 
                        [knowledge , comprehension , application , analysis, synthesis , evaluation]
                    previous response : {reply}""",
                }
            ],
            model="openai/gpt-oss-120b",
        )

        reply = chat_completion.choices[0].message.content.lower()

    pred_labels.append(reply.lower())

100%|██████████| 126/126 [04:26<00:00,  2.12s/it]


In [60]:
print(classification_report(label , [label_mapper[key.lower()] for key in pred_labels]))

              precision    recall  f1-score   support

           0       0.85      1.00      0.92        22
           1       0.88      0.70      0.78        20
           2       0.47      0.47      0.47        15
           3       0.87      0.68      0.76        19
           4       0.72      0.90      0.80        29
           5       0.94      0.81      0.87        21

    accuracy                           0.79       126
   macro avg       0.79      0.76      0.77       126
weighted avg       0.80      0.79      0.78       126



## LLAMA4-Scout

In [ ]:
# Test

query = 'How many total disk access is needed to search a record using two level indexing?'
chat_completion = groq_client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": f"""Classify the query based on Bloom's taxonomy level using ONLY one word from: 
                [knowledge, comprehension, application, analysis, synthesis, evaluation]
            
            query : {query}""",
        }
    ],
    model="meta-llama/llama-4-scout-17b-16e-instruct",
)

reply = chat_completion.choices[0].message.content.lower()

while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
    print(reply)
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""Extract the blooms level from previous reponse. Answer only in one word without punctuation from: 
                    [knowledge , comprehension , application , analysis, synthesis , evaluation]
                previous response : {reply}""",
            }
        ],
        model="meta-llama/llama-4-scout-17b-16e-instruct",
    )

    reply = chat_completion.choices[0].message.content.lower()
    print(reply)


print(reply)

the query requires the test-taker to recall or calculate a specific value related to a concept (two-level indexing). 

the correct classification is: **knowledge**
knowledge
knowledge


### Assign Labels

In [62]:
pred_labels = []

for query in tqdm(queries):
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""Classify the query based on Bloom's taxonomy level using ONLY one word from: 
                    [knowledge, comprehension, application, analysis, synthesis, evaluation]
                
                query : {query}""",
            }
        ],
        model="meta-llama/llama-4-scout-17b-16e-instruct",
    )

    reply = chat_completion.choices[0].message.content.lower()

    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": f"""Extract the blooms level from previous reponse. Answer only in one word without punctuation from: 
                        [knowledge , comprehension , application , analysis, synthesis , evaluation]
                    previous response : {reply}""",
                }
            ],
            model="meta-llama/llama-4-scout-17b-16e-instruct",
        )

        reply = chat_completion.choices[0].message.content.lower()

    pred_labels.append(reply.lower())

100%|██████████| 126/126 [07:11<00:00,  3.42s/it]


In [63]:
print(classification_report(label , [label_mapper[key.lower()] for key in pred_labels]))

              precision    recall  f1-score   support

           0       0.84      0.95      0.89        22
           1       0.80      0.80      0.80        20
           2       0.50      0.47      0.48        15
           3       0.88      0.74      0.80        19
           4       0.72      0.90      0.80        29
           5       0.93      0.67      0.78        21

    accuracy                           0.78       126
   macro avg       0.78      0.75      0.76       126
weighted avg       0.79      0.78      0.77       126



# FEW-Shot